# Apple - Music Discovery Performance for New Artists

In [1]:
import pandas as pd
import numpy as np
import polars as pl
from datetime import date

In [4]:
df_practica = pd.read_csv('../Data/021/practica_public_fct_rtist_recommendations.csv', parse_dates=['recommendation_date'])

pl_practica = pl.read_csv('../Data/021/practica_public_fct_rtist_recommendations.csv').with_columns(pl.col('recommendation_date').str.to_date("%Y-%m-%d"))

# Pregunta 1

### ¿Cuántos artistas únicos fueron recomendados a los usuarios en abril de 2024? Este análisis ayudará a determinar la diversidad de las recomendaciones durante ese mes.

```SQL
SELECT
    COUNT(DISTINCT artist_id) AS total_artistas_unicos
FROM fct_artist_recommendations
WHERE ((EXTRACT(MONTH FROM recommendation_date) = 4) AND
       (EXTRACT(YEAR FROM recommendation_date) = 2024));
```

In [11]:
abril = df_practica[
    (df_practica['recommendation_date'].dt.month == 4) &
    (df_practica['recommendation_date'].dt.year == 2024)
].reset_index()

res = abril['artist_id'].nunique()

res

4

In [16]:
res = pl_practica.filter(
    (pl.col('recommendation_date').dt.month() == 4) &
    (pl.col('recommendation_date').dt.year() == 2024)
).select(
    pl.col('artist_id').n_unique()
)

# Pregunta 2

### ¿Cuál es el número total de recomendaciones de nuevos artistas en mayo de 2024? Esta información ayudará a evaluar si nuestro enfoque en el talento emergente está funcionando de manera efectiva.

```SQL
SELECT
    COUNT(DISTINCT recommendation_id) AS total_recommendation_new_artist
FROM fct_artist_recommendations
WHERE ((EXTRACT(MONTH FROM recommendation_date) = 5) AND
       (EXTRACT(YEAR FROM recommendation_date) = 2024)) AND
    is_new_artist = TRUE;
```

In [21]:
res = df_practica[
    (df_practica['recommendation_date'].dt.month == 5) &
    (df_practica['recommendation_date'].dt.year == 2024) &
    (df_practica['is_new_artist'] == True)
].shape[0]

res

4

In [ ]:
res = pl_practica.filter(
    (pl.col('recommendation_date').dt.month() == 5) &
    (pl.col('recommendation_date').dt.year() == 2024) &
    (pl.col('is_new_artist') == True)
).height

res

4

# Pregunta 3

### Para cada mes del segundo trimestre de 2024 (de abril a junio de 2024), ¿cuántos nuevos artistas distintos fueron recomendados a los usuarios? Este desglose ayudará a identificar tendencias en las recomendaciones de nuevos artistas a lo largo del trimestre.

```SQL
SELECT
    EXTRACT(MONTH FROM recommendation_date) AS mes,
    COUNT(DISTINCT artist_id) AS nuevos_artistas_unicos
FROM fct_artist_recommendations
WHERE (recommendation_date BETWEEN '2024-04-01' AND '2024-06-30') AND
      is_new_artist = TRUE
GROUP BY mes
ORDER BY mes;
```

In [27]:
df_q2 = df_practica[
    (df_practica['recommendation_date'].between('2024-04-01','2024-06-30')) &
    (df_practica['is_new_artist'] == True)
].reset_index()

res = df_q2.groupby(df_q2['recommendation_date'].dt.month)['artist_id'].nunique().reset_index()

res.columns = ['mes','nuevos_artistas_unicos']

res

,mes,nuevos_artistas_unicos
0,4,3
1,5,3
2,6,3


In [31]:
res = pl_practica.filter(
    (pl.col('recommendation_date').is_between(date(2024,4,1),date(2024,6,30))) &
    (pl.col('is_new_artist') == True)
).group_by(
    pl.col('recommendation_date').dt.month().alias('mes')
).agg(
    pl.col('artist_id').n_unique().alias('nuevos_artistas_unicos')
).sort('mes')

res

mes,nuevos_artistas_unicos
i8,u32
4,3
5,3
6,3
